# Lab | Hypothesis Testing

**Objective**

Welcome to the Hypothesis Testing Lab, where we embark on an enlightening journey through the realm of statistical decision-making! In this laboratory, we delve into various scenarios, applying the powerful tools of hypothesis testing to scrutinize and interpret data.

From testing the mean of a single sample (One Sample T-Test), to investigating differences between independent groups (Two Sample T-Test), and exploring relationships within dependent samples (Paired Sample T-Test), our exploration knows no bounds. Furthermore, we'll venture into the realm of Analysis of Variance (ANOVA), unraveling the complexities of comparing means across multiple groups.

So, grab your statistical tools, prepare your hypotheses, and let's embark on this fascinating journey of exploration and discovery in the world of hypothesis testing!

**Challenge 1**

In this challenge, we will be working with pokemon data. The data can be found here:

- https://raw.githubusercontent.com/data-bootcamp-v4/data/main/pokemon.csv

In [1]:
#libraries
import pandas as pd
import scipy.stats as st
import numpy as np



In [2]:
df = pd.read_csv("https://raw.githubusercontent.com/data-bootcamp-v4/data/main/pokemon.csv")
df

,Name,Type 1,Type 2,HP,Attack,Defense,Sp. Atk,Sp. Def,Speed,Generation,Legendary
0,Bulbasaur,Grass,Poison,45,49,49,65,65,45,1,False
1,Ivysaur,Grass,Poison,60,62,63,80,80,60,1,False
2,Venusaur,Grass,Poison,80,82,83,100,100,80,1,False
3,Mega Venusaur,Grass,Poison,80,100,123,122,120,80,1,False
4,Charmander,Fire,NaN,39,52,43,60,50,65,1,False
...,...,...,...,...,...,...,...,...,...,...,...
795,Diancie,Rock,Fairy,50,100,150,100,150,50,6,True
796,Mega Diancie,Rock,Fairy,50,160,110,160,110,110,6,True
797,Hoopa Confined,Psychic,Ghost,80,110,60,150,130,70,6,True
798,Hoopa Unbound,Psychic,Dark,80,160,60,170,130,80,6,True


- We posit that Pokemons of type Dragon have, on average, more HP stats than Grass. Choose the propper test and, with 5% significance, comment your findings.

Define the hypotheses:
H₀: Dragon and Grass pokemon have the same average HP
H₁: Dragon pokemon have MORE average HP than Grass pokemon

In [3]:
dragon = df[df['Type 1'] == 'Dragon']['HP']
grass = df[df['Type 1'] == 'Grass']['HP']

print(f"Dragon pokemon: {len(dragon)}")
print(f"Grass pokemon: {len(grass)}")

Dragon pokemon: 32
Grass pokemon: 70


In [4]:
# Normality test
stat_dragon, p_dragon = st.normaltest(dragon)
stat_grass, p_grass = st.normaltest(grass)

print(f"Dragon normality p-value: {p_dragon:.4f}")
print(f"Grass normality p-value: {p_grass:.4f}")

# Equal variances
stat_levene, p_levene = st.levene(dragon, grass)
print(f"Levene's p-value: {p_levene:.4f}")

Dragon normality p-value: 0.5940
Grass normality p-value: 0.2224
Levene's p-value: 0.1748


p = bigger than 0.05 in all 3 cases so all 3 prerequisities are met.

In [5]:
# One-sided t-test (Dragons > Grass)
stat, p_value = st.ttest_ind(dragon, grass, 
                                 equal_var=True, 
                                 alternative='greater')

print(f"T-statistic: {stat:.4f}")
print(f"P-value: {p_value:.4f}")

# Conclusion
alpha = 0.05
if p_value < alpha:
    print("Reject H₀ — Dragon pokemon have significantly more HP than Grass!")
else:
    print("Fail to reject H₀ — No significant difference in HP between Dragon and Grass")

T-statistic: 3.5904
P-value: 0.0003
Reject H₀ — Dragon pokemon have significantly more HP than Grass!


- We posit that Legendary Pokemons have different stats (HP, Attack, Defense, Sp.Atk, Sp.Def, Speed) when comparing with Non-Legendary. Choose the propper test and, with 5% significance, comment your findings.


In [6]:
#split data
legendary = df[df['Legendary'] == True]
non_legendary = df[df['Legendary'] == False]

print(f"Legendary pokemon: {len(legendary)}")
print(f"Non-Legendary pokemon: {len(non_legendary)}")

Legendary pokemon: 65
Non-Legendary pokemon: 735


In [7]:
#check prerequisites for all 6 stats at once:
stats_to_test = ['HP', 'Attack', 'Defense', 'Sp. Atk', 'Sp. Def', 'Speed']

for stat in stats_to_test:
    leg = legendary[stat]
    non_leg = non_legendary[stat]
    
    # Normality
    _, p_leg = st.normaltest(leg)
    _, p_non_leg = st.normaltest(non_leg)
    
    # Equal variances
    _, p_levene = st.levene(leg, non_leg)
    
    print(f"\n{stat}:")
    print(f"  Legendary normality p-value: {p_leg:.4f}")
    print(f"  Non-Legendary normality p-value: {p_non_leg:.4f}")
    print(f"  Levene's p-value: {p_levene:.4f}")


HP:
  Legendary normality p-value: 0.4507
  Non-Legendary normality p-value: 0.0000
  Levene's p-value: 0.5758

Attack:
  Legendary normality p-value: 0.2256
  Non-Legendary normality p-value: 0.0000
  Levene's p-value: 0.9945

Defense:
  Legendary normality p-value: 0.0029
  Non-Legendary normality p-value: 0.0000
  Levene's p-value: 0.3203

Sp. Atk:
  Legendary normality p-value: 0.6691
  Non-Legendary normality p-value: 0.0000
  Levene's p-value: 0.3701

Sp. Def:
  Legendary normality p-value: 0.0340
  Non-Legendary normality p-value: 0.0000
  Levene's p-value: 0.7764

Speed:
  Legendary normality p-value: 0.0142
  Non-Legendary normality p-value: 0.0000
  Levene's p-value: 0.0018


Non-legendary fails normality for ALL stats → but with 735 pokemon CLT saves us! 
Legendary fails normality for Defense, Sp. Def and Speed → but 65 pokemon is close enough to rely on CLT 
Speed has unequal variances → we use equal_var=False for Speed only!

In [9]:
stats_to_test = ['HP', 'Attack', 'Defense', 'Sp. Atk', 'Sp. Def', 'Speed']
alpha = 0.05

for stat in stats_to_test:
    leg = legendary[stat]
    non_leg = non_legendary[stat]
    
    # Use equal_var=False for Speed, True for rest
    equal_var = False if stat == 'Speed' else True
    
    t_stat, p_value = st.ttest_ind(leg, non_leg,
                                      equal_var=equal_var,
                                      alternative='two-sided')
    
    conclusion = "Reject H₀ ✅" if p_value < alpha else "Fail to reject H₀ ❌"
    
    print(f"{stat}:")
    print(f"  T-statistic: {t_stat:.4f}")
    print(f"  P-value: {p_value:.4f}")
    print(f"  {conclusion}")
    print()

HP:
  T-statistic: 8.0361
  P-value: 0.0000
  Reject H₀ ✅

Attack:
  T-statistic: 10.3973
  P-value: 0.0000
  Reject H₀ ✅

Defense:
  T-statistic: 7.1812
  P-value: 0.0000
  Reject H₀ ✅

Sp. Atk:
  T-statistic: 14.1914
  P-value: 0.0000
  Reject H₀ ✅

Sp. Def:
  T-statistic: 11.0378
  P-value: 0.0000
  Reject H₀ ✅

Speed:
  T-statistic: 11.4750
  P-value: 0.0000
  Reject H₀ ✅



With a significance level of 0.05, we reject H₀ for all 6 stats — Legendary pokemon have significantly different (higher) stats than Non-Legendary pokemon across every single category!

**Challenge 2**

In this challenge, we will be working with california-housing data. The data can be found here:
- https://raw.githubusercontent.com/data-bootcamp-v4/data/main/california_housing.csv

In [10]:
df = pd.read_csv("https://raw.githubusercontent.com/data-bootcamp-v4/data/main/california_housing.csv")
df.head()

,longitude,latitude,housing_median_age,total_rooms,total_bedrooms,population,households,median_income,median_house_value
0,-114.31,34.19,15.0,5612.0,1283.0,1015.0,472.0,1.4936,66900.0
1,-114.47,34.40,19.0,7650.0,1901.0,1129.0,463.0,1.8200,80100.0
2,-114.56,33.69,17.0,720.0,174.0,333.0,117.0,1.6509,85700.0
3,-114.57,33.64,14.0,1501.0,337.0,515.0,226.0,3.1917,73400.0
4,-114.57,33.57,20.0,1454.0,326.0,624.0,262.0,1.9250,65500.0


**We posit that houses close to either a school or a hospital are more expensive.**

- School coordinates (-118, 34)
- Hospital coordinates (-122, 37)

We consider a house (neighborhood) to be close to a school or hospital if the distance is lower than 0.50.

Hint:
- Write a function to calculate euclidean distance from each house (neighborhood) to the school and to the hospital.
- Divide your dataset into houses close and far from either a hospital or school.
- Choose the propper test and, with 5% significance, comment your findings.
 

In [11]:
# Coordinates
school = (-118, 34)
hospital = (-122, 37)

# Euclidean distance function
def euclidean_distance(lat, lon, point):
    return np.sqrt((lon - point[0])**2 + (lat - point[1])**2)

# Calculate distances for each house
df['distance_school'] = euclidean_distance(df['latitude'], df['longitude'], school)
df['distance_hospital'] = euclidean_distance(df['latitude'], df['longitude'], hospital)

# Check results
print(df[['latitude', 'longitude', 'distance_school', 'distance_hospital']].head())

   latitude  longitude  distance_school  distance_hospital
0     34.19    -114.31         3.694888           8.187319
1     34.40    -114.47         3.552591           7.966235
2     33.69    -114.56         3.453940           8.143077
3     33.64    -114.57         3.448840           8.154416
4     33.57    -114.57         3.456848           8.183508


In [12]:
# A house is 'close' if distance to school OR hospital is less than 0.50
df['close'] = (df['distance_school'] < 0.50) | (df['distance_hospital'] < 0.50)

# Split into two groups
close = df[df['close'] == True]['median_house_value']
far = df[df['close'] == False]['median_house_value']

print(f"Houses close to school or hospital: {len(close)}")
print(f"Houses far from school and hospital: {len(far)}")

Houses close to school or hospital: 6829
Houses far from school and hospital: 10171


In [13]:
# Check normality
stat_close, p_close = st.normaltest(close)
stat_far, p_far = st.normaltest(far)

print(f"Close normality p-value: {p_close:.4f}")
print(f"Far normality p-value: {p_far:.4f}")

# Check equal variances
stat_levene, p_levene = st.levene(close, far)
print(f"Levene's p-value: {p_levene:.4f}")

Close normality p-value: 0.0000
Far normality p-value: 0.0000
Levene's p-value: 0.0208


Test                        P-value Result
Close normality             0.0000  ❌ Not normal
Far normality               0.0000  ❌ Not normal
Levene's equal variances    0.0208  ❌ Unequal variances

But:
Normality fails → but CLT saves us (6,829 and 10,171 are huge samples!) ✅
Equal variances fails → we use equal_var=False (Welch's T-test) ✅

In [14]:
# One-sided t-test (close > far)
t_stat, p_value = st.ttest_ind(close, far,
                                   equal_var=False,
                                   alternative='greater')

alpha = 0.05
print(f"T-statistic: {t_stat:.4f}")
print(f"P-value: {p_value:.4f}")

if p_value < alpha:
    print("Reject H₀ ✅ — houses close to school/hospital ARE more expensive!")
else:
    print("Fail to reject H₀ ❌ — no significant difference in price!")

T-statistic: 37.9923
P-value: 0.0000
Reject H₀ ✅ — houses close to school/hospital ARE more expensive!


In [15]:
print(f"Average price close to school/hospital: ${close.mean():,.2f}")
print(f"Average price far from school/hospital: ${far.mean():,.2f}")
print(f"Difference: ${close.mean() - far.mean():,.2f}")

Average price close to school/hospital: $246,951.98
Average price far from school/hospital: $180,678.44
Difference: $66,273.54


Conclusion:
With a T-statistic of 37.99 and a p-value of essentially 0, we reject H₀ with 95% confidence. 
Houses close to a school or hospital are on average $66,273 more expensive than houses far from both — a difference of about 37%! 🎯